# Email-Discovery Model Benchmark

Compares web-search **discovery backends** head-to-head on the same 300 podcast leads,
scored against the human-verified ground truth (`final_human` tab).

**What's held constant** (so this is an apples-to-apples *backend* test, not a prompt test):
- the exact lead context fed in (name + website + socials)
- the exact discovery prompt (your production `perplexity_email_discovery` prompt, verbatim)
- the JSON output contract + candidate ranking

**What varies:** the agentic-search backend — OpenAI (web_search), Gemini (grounding),
Claude (web_search), each compared against your existing **Perplexity pro-search** run
(read from the `final_agent` tab — no re-run needed).

**Scoring (tiered, against `final_human`):**
| bucket | meaning | maps to your existing `email_comparison` |
|---|---|---|
| `exact` | found ∈ ground-truth set | Both found and same |
| `domain_only` | same domain, different mailbox | Both found but different (right org) |
| `different` | found, wrong domain | Both found but different (wrong) |
| `missed` | GT has email, model found none | Human found but agent could not |
| `found_no_gt` | model found one, GT has none | Agent found but human could not |
| `correct_abstain` | GT has none, model found none | Both could not find |

> Run order: run cell-by-cell. **Cell 6 does a dry run first** (no API spend). Set `LIMIT=None`
> and re-run the runner cell for the full 300. Set `WRITE_TO_SHEET=True` to publish `Benchmark_*` tabs.


In [ ]:
from __future__ import annotations
import os, re, json, time, html
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

import sys
sys.path.insert(0, "/Users/utkarshumang/my_projects/lead-enricher-ai-be")   # google_utils
sys.path.insert(0, "/Users/utkarshumang/my_projects/ai-agents-service")

from dotenv import load_dotenv
load_dotenv("/Users/utkarshumang/my_projects/lead-enricher-ai-be/.env")

import pandas as pd
from google_utils.google_sheet import GoogleSheetService

# ── Config ──────────────────────────────────────────────────────────────────
SPREADSHEET_URL  = "https://docs.google.com/spreadsheets/d/10D3OJY9fh5Gk4leWC7mF5iZ5_ZsY9LaFwksrrAT7S_Q/edit"
GROUND_TRUTH_TAB = "final_human"     # human-verified Podcast Email = ground truth
BASELINE_TAB     = "final_agent"     # existing Perplexity pro-search run
NAME_COL         = "Podcast Name"
GT_EMAIL_COL     = "Podcast Email"

# Which backends to benchmark. Each auto-skips if its API key is missing.
RUN_OPENAI       = True
RUN_GEMINI       = True
RUN_CLAUDE       = True            # needs ANTHROPIC_API_KEY (not in .env yet → will skip)
RERUN_PERPLEXITY = False           # False = use existing final_agent column as the Perplexity baseline

OPENAI_MODEL     = "gpt-5.1"
GEMINI_MODEL     = "gemini-2.5-pro"
CLAUDE_MODEL     = "anthropic/claude-opus-4-8"   # litellm id; set the deployed id you have a key for
PERPLEXITY_PRESET = "pro-search"

LIMIT            = 10              # small test first; set None for all 300
CONCURRENCY      = 5
WRITE_TO_SHEET   = False
CHECKPOINT_FILE  = "benchmark_discovery_checkpoint.json"
DETAIL_CSV       = "benchmark_discovery_detail.csv"

OPENAI_API_KEY     = os.environ.get("OPENAI_API_KEY", "")
GEMINI_API_KEY     = os.environ.get("GEMINI_API_KEY", "")
PERPLEXITY_API_KEY = os.environ.get("PERPLEXITY_API_KEY", "")
ANTHROPIC_API_KEY  = os.environ.get("ANTHROPIC_API_KEY", "")
print("keys present:", {k: bool(v) for k, v in {
    "openai": OPENAI_API_KEY, "gemini": GEMINI_API_KEY,
    "perplexity": PERPLEXITY_API_KEY, "anthropic": ANTHROPIC_API_KEY}.items()})


In [ ]:
sheet = GoogleSheetService()
SPREADSHEET_ID = sheet.extract_spreadsheet_id(SPREADSHEET_URL)

ok, gt_df = sheet.get_sheet_data(SPREADSHEET_ID, f"{GROUND_TRUTH_TAB}!A1:Z1000");  assert ok, gt_df
ok, base_df = sheet.get_sheet_data(SPREADSHEET_ID, f"{BASELINE_TAB}!A1:Z1000");     assert ok, base_df

def clean(x):  return str(x or "").strip()
def unesc(x):  return html.unescape(clean(x))

def parse_emails(cell):
    out, seen = [], set()
    for part in re.split(r"[,\s;]+", clean(cell)):
        e = part.strip().lower().rstrip(".,);]")
        if "@" in e and "." in e.split("@")[-1] and e not in seen:
            seen.add(e); out.append(e)
    return out

baseline_by_name = {
    unesc(r[NAME_COL]): {
        "email": clean(r.get("Email Found", "")),
        "status": clean(r.get("Status", "")),
    } for _, r in base_df.iterrows()
}

leads = []
for _, r in gt_df.iterrows():
    name = unesc(r.get(NAME_COL, ""))
    if not name:
        continue
    leads.append({
        "name": name,
        "website":   clean(r.get("Podcast Website", "")),
        "facebook":  clean(r.get("Podcast Facebook", "")),
        "twitter":   clean(r.get("Podcast Twitter", "")),
        "instagram": clean(r.get("Podcast Instagram", "")),
        "youtube":   clean(r.get("Podcast YouTube", "")),
        "linkedin":  clean(r.get("Podcast LinkedIn", "")),
        "gt_emails": parse_emails(r.get(GT_EMAIL_COL, "")),
    })

if LIMIT:
    leads = leads[:LIMIT]
n_gt = sum(1 for l in leads if l["gt_emails"])
print(f"{len(leads)} leads · {n_gt} have ground-truth email · {len(leads) - n_gt} have none")


In [ ]:
PERPLEXITY_PROMPT = """Find the contact email address for the following person or entity. Search their website, Linktree, social media bios, and any other public sources.

{{available_info}}

Search thoroughly — check their personal site, company site, any link-in-bio pages, podcast directory listings (Apple Podcasts, Spotify, Listen Notes), and social media profiles.

IMPORTANT: If the source_type is "podscan_guest", the target is the GUEST — find the guest's personal or company email, NOT the podcast host's email. The podcast name is context to help identify the person but the email must belong to the guest or their company.

Email quality rules (apply before including any email):
- Exclude tagged/plus-addressed emails such as info+xyz@domain.com or contact+podcast@domain.com — the presence of a "+" in the local part is a strong signal that this is a filtered alias, not a real contact address.
- Exclude generic platform no-reply addresses (noreply@, donotreply@, mailer@).
- Prefer a personal or show-specific address over generic prefixes (info@, contact@, hello@, support@, admin@, team@) when both are available.

Return ONLY a JSON object, no other text:
{
    "emails_found": [
        {
            "email": "string",
            "source": "string — where exactly you found it",
            "confidence": 0.0 to 1.0,
            "note": "string or null"
        }
    ],
    "search_summary": "brief summary of what you searched and what you found",
    "not_found_reason": "string — only present when emails_found is empty; explain specifically what you searched and why no email was found (e.g. 'Website has a contact form only, no email address displayed. No email found in podcast RSS feed, Apple Podcasts listing, or social media bios.')"
}

Confidence guide:
- 1.0 — on their personal website or LinkedIn profile
- 0.8 — on podcast/channel contact page or RSS feed
- 0.6 — in social media bio or Linktree
- 0.4 — in a third-party listing or article
- 0.2 — inferred or uncertain"""

def build_available_info(lead):
    parts = [f"Name: {lead['name']}", "Type: Podcast"]
    if lead["website"]:
        parts.append(f"Website: {lead['website']}")
    for label, key in [("YouTube", "youtube"), ("LinkedIn", "linkedin"),
                       ("Twitter", "twitter"), ("Instagram", "instagram"),
                       ("Facebook", "facebook")]:
        if lead[key]:
            parts.append(f"{label}: {lead[key]}")
    return "\n".join(parts)

def build_prompt(lead):
    # keep JSON braces in the template intact — only replace the {{available_info}} slot
    return PERPLEXITY_PROMPT.replace("{{available_info}}", build_available_info(lead))

print(build_prompt(leads[0])[:600])


In [ ]:
def extract_json(text):
    if not text:
        return None
    t = re.sub(r"^```json|^```|```$", "", text.strip(), flags=re.MULTILINE).strip()
    start = t.find("{")
    if start < 0:
        return None
    depth = 0
    for i in range(start, len(t)):
        if t[i] == "{":
            depth += 1
        elif t[i] == "}":
            depth -= 1
            if depth == 0:
                try:
                    return json.loads(t[start:i + 1])
                except Exception:
                    break
    try:
        return json.loads(t)
    except Exception:
        return None

_GENERIC = {"info", "contact", "hello", "support", "admin", "team", "mail",
            "enquiries", "enquiry", "podcast", "podcasts"}

def pick_best(data):
    """Return (email, confidence, source) — best candidate, generics penalised, plus-addr dropped."""
    if not data:
        return None, 0.0, ""
    cands = []
    for it in (data.get("emails_found") or []):
        e = clean(it.get("email")).lower()
        if "@" not in e or "+" in e.split("@")[0]:
            continue
        try:
            conf = float(it.get("confidence", 0.5) or 0)
        except (TypeError, ValueError):
            conf = 0.5
        cands.append((e, conf, clean(it.get("source"))))
    if not cands:
        return None, 0.0, ""
    cands.sort(key=lambda c: c[1] - (0.2 if c[0].split("@")[0] in _GENERIC else 0.0), reverse=True)
    return cands[0]


In [ ]:
def _err(msg):
    return {"email": None, "confidence": 0.0, "source": "", "raw": "", "error": str(msg)[:300],
            "in_tok": 0, "out_tok": 0}

def _ok(text, in_tok, out_tok):
    e, c, s = pick_best(extract_json(text))
    return {"email": e, "confidence": c, "source": s, "raw": (text or "")[:1500],
            "error": None, "in_tok": in_tok or 0, "out_tok": out_tok or 0}

def discover_openai(prompt):
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)
    last = None
    for tool in ({"type": "web_search"}, {"type": "web_search_preview"}):
        try:
            r = client.responses.create(model=OPENAI_MODEL, tools=[tool], input=prompt)
            u = getattr(r, "usage", None)
            return _ok(r.output_text, getattr(u, "input_tokens", 0), getattr(u, "output_tokens", 0))
        except Exception as ex:
            last = ex
    return _err(last)

def discover_gemini(prompt):
    from google import genai
    from google.genai import types
    try:
        client = genai.Client(api_key=GEMINI_API_KEY)
        r = client.models.generate_content(
            model=GEMINI_MODEL, contents=prompt,
            config=types.GenerateContentConfig(
                tools=[types.Tool(google_search=types.GoogleSearch())]))
        u = getattr(r, "usage_metadata", None)
        return _ok(r.text, getattr(u, "prompt_token_count", 0), getattr(u, "candidates_token_count", 0))
    except Exception as ex:
        return _err(ex)

def discover_claude(prompt):
    import litellm
    try:
        r = litellm.completion(
            model=CLAUDE_MODEL, api_key=ANTHROPIC_API_KEY,
            messages=[{"role": "user", "content": prompt}],
            tools=[{"type": "web_search_20250305", "name": "web_search", "max_uses": 5}])
        msg = r.choices[0].message
        if isinstance(msg.content, str):
            text = msg.content
        else:
            text = " ".join(b.get("text", "") for b in (msg.content or []) if isinstance(b, dict))
        u = getattr(r, "usage", None)
        return _ok(text, getattr(u, "prompt_tokens", 0), getattr(u, "completion_tokens", 0))
    except Exception as ex:
        return _err(ex)

def discover_perplexity(prompt):
    import httpx
    try:
        with httpx.Client(timeout=120.0) as cl:
            resp = cl.post(
                "https://api.perplexity.ai/v1/agent",
                headers={"Authorization": f"Bearer {PERPLEXITY_API_KEY}", "Content-Type": "application/json"},
                json={"preset": PERPLEXITY_PRESET, "input": prompt,
                      "instructions": "Return ONLY a valid JSON object. No citation markers, no markdown fences."})
            resp.raise_for_status()
            data = resp.json()
        text = "".join(b.get("text", "") for it in data.get("output", [])
                       for b in it.get("content", []) if b.get("type") == "output_text")
        u = data.get("usage") or {}
        return _ok(text, u.get("prompt_tokens") or u.get("input_tokens"),
                   u.get("completion_tokens") or u.get("output_tokens"))
    except Exception as ex:
        return _err(ex)

# pricing USD per 1M tokens + flat per-call web-search surcharge — EDIT to match your billing.
PRICING = {
    "openai":     {"in": 1.25, "out": 10.0, "per_call": 0.010},
    "gemini":     {"in": 1.25, "out": 10.0, "per_call": 0.035},
    "claude":     {"in": 15.0, "out": 75.0, "per_call": 0.010},
    "perplexity": {"in": 1.0,  "out": 1.0,  "per_call": 0.005},
}
def est_cost(model, in_tok, out_tok):
    p = PRICING.get(model)
    return 0.0 if not p else round(in_tok / 1e6 * p["in"] + out_tok / 1e6 * p["out"] + p["per_call"], 5)

BACKENDS = {}
if RUN_OPENAI and OPENAI_API_KEY:           BACKENDS["openai"] = discover_openai
if RUN_GEMINI and GEMINI_API_KEY:           BACKENDS["gemini"] = discover_gemini
if RUN_CLAUDE and ANTHROPIC_API_KEY:        BACKENDS["claude"] = discover_claude
if RERUN_PERPLEXITY and PERPLEXITY_API_KEY: BACKENDS["perplexity"] = discover_perplexity

skipped = [m for m, on in [("openai", RUN_OPENAI), ("gemini", RUN_GEMINI),
                           ("claude", RUN_CLAUDE)] if on and m not in BACKENDS]
print("active backends:", list(BACKENDS) or "(none!)")
if skipped:
    print("skipped (missing key):", skipped)
print("perplexity baseline:", "re-run" if "perplexity" in BACKENDS else f"read from '{BASELINE_TAB}' tab")


In [ ]:
def _load_ckpt():
    return json.load(open(CHECKPOINT_FILE)) if os.path.exists(CHECKPOINT_FILE) else {}
def _save_ckpt(c):
    json.dump(c, open(CHECKPOINT_FILE, "w"), indent=2)

def run_benchmark(dry_run=False):
    ckpt = _load_ckpt()
    pending = [(f"{m}||{l['name']}", m, fn, l)
               for l in leads for m, fn in BACKENDS.items()
               if f"{m}||{l['name']}" not in ckpt]
    cached = len(BACKENDS) * len(leads) - len(pending)
    print(f"{len(pending)} calls pending · {cached} cached · "
          f"{len(BACKENDS)} models × {len(leads)} leads")
    if dry_run:
        print("DRY RUN — no API calls made. Set dry_run=False to execute.")
        return ckpt

    def work(item):
        key, model, fn, lead = item
        t0 = time.time()
        res = fn(build_prompt(lead))
        res["latency"] = round(time.time() - t0, 2)
        res["cost"] = est_cost(model, res.get("in_tok", 0), res.get("out_tok", 0))
        return key, res

    done = 0
    with ThreadPoolExecutor(max_workers=CONCURRENCY) as ex:
        futures = [ex.submit(work, it) for it in pending]
        for f in as_completed(futures):
            key, res = f.result()
            ckpt[key] = res
            done += 1
            if done % 10 == 0 or done == len(pending):
                _save_ckpt(ckpt)
                print(f"  {done}/{len(pending)} done")
    _save_ckpt(ckpt)
    print("done.")
    return ckpt

_ = run_benchmark(dry_run=True)


## Run the benchmark (live)

This spends API credits. Start small (`LIMIT=10` from the config cell), confirm the scorecard
looks right, then set `LIMIT=None` and re-run this cell for the full 300. The checkpoint file
makes it resumable — re-running only fills in missing (model, lead) pairs.

In [ ]:
results = run_benchmark(dry_run=False)
print(f"\ntotal cached results: {len(results)}")


In [ ]:
def _domain(e):
    return e.split("@")[-1].lower() if e and "@" in e else ""

def classify(found, gt_set):
    f = clean(found).lower()
    if not gt_set:
        return "found_no_gt" if f else "correct_abstain"
    if not f:
        return "missed"
    if f in gt_set:
        return "exact"
    if _domain(f) in {_domain(g) for g in gt_set}:
        return "domain_only"
    return "different"

results = _load_ckpt()
models = list(BACKENDS)
include_pplx_baseline = "perplexity" not in models  # read from final_agent tab

rows = []
for lead in leads:
    gt = set(lead["gt_emails"])
    rec = {"name": lead["name"], "gt": ", ".join(sorted(gt)) or "(none)"}
    for m in models:
        r = results.get(f"{m}||{lead['name']}", {})
        rec[f"{m}_email"] = r.get("email")
        rec[f"{m}_cat"]   = classify(r.get("email"), gt)
        rec[f"{m}_cost"]  = r.get("cost", 0.0)
        rec[f"{m}_lat"]   = r.get("latency", 0.0)
    if include_pplx_baseline:
        be = parse_emails(baseline_by_name.get(lead["name"], {}).get("email", ""))
        bemail = be[0] if be else None
        rec["perplexity_email"] = bemail
        rec["perplexity_cat"]   = classify(bemail, gt)
        rec["perplexity_cost"]  = 0.0
        rec["perplexity_lat"]   = 0.0
    rows.append(rec)

detail = pd.DataFrame(rows)
detail.to_csv(DETAIL_CSV, index=False)
all_models = models + (["perplexity"] if include_pplx_baseline else [])

CATS = ["exact", "domain_only", "different", "missed", "found_no_gt", "correct_abstain"]
n_gt = sum(1 for l in leads if l["gt_emails"])
n_nogt = len(leads) - n_gt

summary = []
for m in all_models:
    counts = {c: 0 for c in CATS}
    cost, lats = 0.0, []
    for _, r in detail.iterrows():
        counts[r[f"{m}_cat"]] += 1
        cost += r.get(f"{m}_cost", 0.0) or 0.0
        if r.get(f"{m}_lat"):
            lats.append(r[f"{m}_lat"])
    exact_rate = counts["exact"] / n_gt * 100 if n_gt else 0
    dom_rate = (counts["exact"] + counts["domain_only"]) / n_gt * 100 if n_gt else 0
    summary.append({
        "model": m,
        "exact": counts["exact"],
        "exact_%": round(exact_rate, 1),
        "domain_only": counts["domain_only"],
        "exact+domain_%": round(dom_rate, 1),
        "wrong": counts["different"],
        "missed": counts["missed"],
        "found_no_GT": counts["found_no_gt"],
        "abstain_ok": counts["correct_abstain"],
        "cost_$": round(cost, 3),
        "med_lat_s": round(sorted(lats)[len(lats) // 2], 1) if lats else 0,
    })

scorecard = pd.DataFrame(summary).set_index("model")
print(f"Ground truth: {n_gt} leads with email, {n_nogt} with none "
      f"(rates below are over the {n_gt} with-email leads)\n")
print(scorecard.to_string())


### Reading the scorecard

- **`exact_%`** is the headline hit-rate: found exactly a human-verified email (over the leads that *have* one).
- **`exact+domain_%`** credits same-domain hits (e.g. `hello@show.com` when GT is `jane@show.com`) — right org, wrong mailbox. For cold outreach this is often still usable.
- **`wrong`** = found an email on the wrong domain (the dangerous bucket — confident but incorrect).
- **`missed`** = gave up where a human found one. **`abstain_ok`** = correctly found nothing where none exists (good — not a failure).
- **`cost_$` / `med_lat_s`** are per-run totals/medians using the editable `PRICING` table — treat as ballpark, refine with your actual billing.

The Perplexity row is your existing `final_agent` baseline (cost shown as 0 because it was already run). Set `RERUN_PERPLEXITY=True` to re-run it through the identical prompt for a perfectly fair cost/latency comparison.

In [ ]:
def _ensure_tab(name):
    ok, names = sheet.list_sheets(SPREADSHEET_ID)
    if ok and name in names:
        return
    sheet.service.spreadsheets().batchUpdate(
        spreadsheetId=SPREADSHEET_ID,
        body={"requests": [{"addSheet": {"properties": {"title": name}}}]},
    ).execute()
    print(f"created tab '{name}'")

if WRITE_TO_SHEET:
    sum_tab = "Benchmark_Summary"
    _ensure_tab(sum_tab)
    hdr = ["model"] + list(scorecard.columns)
    data = [hdr] + [[idx] + [r[c] for c in scorecard.columns] for idx, r in scorecard.iterrows()]
    data += [[], ["Generated", datetime.now().strftime("%Y-%m-%d %H:%M"), f"{len(leads)} leads"]]
    print(sheet.clear_and_rewrite_sheet(SPREADSHEET_ID, sum_tab, data)[1])

    for m in all_models:
        tab = f"Benchmark_{m}"
        _ensure_tab(tab)
        d = [["Podcast Name", "Ground Truth", "Email Found", "Category", "Cost $", "Latency s"]]
        for _, r in detail.iterrows():
            d.append([r["name"], r["gt"], r.get(f"{m}_email") or "",
                      r[f"{m}_cat"], r.get(f"{m}_cost", 0), r.get(f"{m}_lat", 0)])
        print(sheet.clear_and_rewrite_sheet(SPREADSHEET_ID, tab, d)[1])
else:
    print("WRITE_TO_SHEET=False — set True to publish Benchmark_Summary + Benchmark_<model> tabs")
